# 🧾 Phase 1 — Synthetic Data Generation
### Personal Spending Intelligence System

**What this notebook does:**
Generates 12 months of realistic, fake personal transaction data and saves it as a CSV file.

**Why synthetic data?**
- Real bank data is private — you can't upload it to GitHub.
- Synthetic data lets you *design* the patterns your ML model will later learn from.
- Being able to generate realistic data is itself a valuable data science skill.

**What you'll learn:**
- How to use Python's `random` and `datetime` libraries
- How to build structured data with `pandas`
- How to save a DataFrame as a CSV


## Step 1 — Import libraries

Think of libraries as toolboxes. We import them once at the top so we can use their tools throughout the notebook.


In [2]:
import pandas as pd          # pandas  → our main tool for working with table-like data (DataFrames)
import numpy as np           # numpy   → math and random number generation
import random                # random  → picking random items from lists
from datetime import date, timedelta  # datetime → working with dates

# Fix the random seed so results are REPRODUCIBLE
# This means every time you run this notebook, you get the exact same data.
# Very important for sharing with others and for debugging.
np.random.seed(42)
random.seed(42)

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


## Step 2 — Define spending categories and their rules

Real spending has patterns. Rent is fixed. Groceries vary. Coffee is frequent and small.
We encode these human behaviours as Python dictionaries and lists.

**Key concept — dictionary:**
A dictionary maps a *key* to a *value*:
```python
{"category_name": {"avg_amount": 50, "std_dev": 10, ...}}
```


In [3]:
# Each category has:
#   avg_amount  → typical spend in ₹
#   std_dev     → how much it varies (std deviation — you'll learn this in EDA)
#   frequency   → how many times per month on average
#   vendors     → realistic merchant names to pick from

CATEGORIES = {
    "Rent": {
        "avg_amount": 18000,
        "std_dev": 0,           # Rent is fixed — no variation
        "frequency": 1,
        "vendors": ["Housing Society", "Landlord Transfer"],
        "day_of_month": 1       # Always paid on the 1st
    },
    "Groceries": {
        "avg_amount": 800,
        "std_dev": 200,
        "frequency": 8,
        "vendors": ["BigBasket", "Zepto", "Swiggy Instamart", "D-Mart", "More Supermarket"]
    },
    "Food & Dining": {
        "avg_amount": 350,
        "std_dev": 150,
        "frequency": 12,
        "vendors": ["Zomato", "Swiggy", "Meghana Foods", "CTR", "Brahmin's Coffee Bar", "Truffles", "Byg Brewski"]
    },
    "Coffee & Snacks": {
        "avg_amount": 180,
        "std_dev": 60,
        "frequency": 14,        # Almost every other day
        "vendors": ["Starbucks", "Blue Tokai", "Third Wave Coffee", "Chaayos", "Cafe Coffee Day"]
    },
    "Transport": {
        "avg_amount": 220,
        "std_dev": 100,
        "frequency": 18,
        "vendors": ["Uber", "Ola", "Rapido", "Namma Metro", "BMTC"]
    },
    "Subscriptions": {
        "avg_amount": 200,
        "std_dev": 20,
        "frequency": 4,         # ~4 subscriptions renewing each month
        "vendors": ["Netflix", "Spotify", "Amazon Prime", "YouTube Premium"],
        "day_of_month": None    # fixed date per sub — handled separately
    },
    "Shopping": {
        "avg_amount": 1200,
        "std_dev": 800,
        "frequency": 3,
        "vendors": ["Amazon", "Flipkart", "Myntra", "Meesho", "Nykaa"]
    },
    "Health & Fitness": {
        "avg_amount": 600,
        "std_dev": 300,
        "frequency": 3,
        "vendors": ["Cult.fit", "PharmEasy", "Apollo Pharmacy", "1mg", "Gym Monthly"]
    },
    "Utilities": {
        "avg_amount": 900,
        "std_dev": 150,
        "frequency": 2,
        "vendors": ["BESCOM", "Airtel Broadband", "Jio Recharge", "BWSSB"]
    },
    "Entertainment": {
        "avg_amount": 700,
        "std_dev": 300,
        "frequency": 2,
        "vendors": ["BookMyShow", "PVR Cinemas", "Wonderla", "Steam", "PlayStation Store"]
    }
}

# Monthly salary credit (income, not expense)
MONTHLY_SALARY = 65000

print(f"✅ Defined {len(CATEGORIES)} spending categories")
print(f"   Categories: {', '.join(CATEGORIES.keys())}")


✅ Defined 10 spending categories
   Categories: Rent, Groceries, Food & Dining, Coffee & Snacks, Transport, Subscriptions, Shopping, Health & Fitness, Utilities, Entertainment


## Step 3 — Generate transactions for a full year

We'll loop through each month of 2024, and for each category, generate the right number of transactions with realistic amounts and dates.

**Key concept — `np.random.normal(mean, std)`:**
This generates a number drawn from a "bell curve" around the mean. Most values will be close to the mean, with occasional high or low outliers — just like real spending!


In [4]:
transactions = []   # We'll collect all transactions here as a list of dicts

START_DATE = date(2024, 1, 1)
END_DATE   = date(2024, 12, 31)

def random_date_in_month(year, month):
    """Pick a random day within a given month."""
    if month == 12:
        last_day = 31
    else:
        last_day = (date(year, month + 1, 1) - timedelta(days=1)).day
    return date(year, month, random.randint(1, last_day))

def generate_amount(avg, std):
    """
    Generate a realistic spend amount.
    - Draw from a normal distribution
    - Round to nearest 10 (feels more like real transactions)
    - Ensure it's never negative or zero
    """
    amount = np.random.normal(avg, std)
    amount = max(amount, avg * 0.2)     # never less than 20% of avg
    return round(amount / 10) * 10      # round to nearest ₹10

# ── Loop through 12 months ────────────────────────────────────────────────────
for month in range(1, 13):
    year = 2024

    # 1. Salary credit on the 1st
    transactions.append({
        "date":        date(year, month, 1),
        "description": "Salary Credit — Connectivity Solutions",
        "amount":      MONTHLY_SALARY,
        "category":    "Income",
        "type":        "credit"
    })

    # 2. Generate expense transactions per category
    for cat_name, cat_info in CATEGORIES.items():
        n = cat_info["frequency"]

        # Add ±1 random variation to frequency (some months you eat out more)
        n = max(1, n + random.randint(-1, 1))

        for _ in range(n):
            # Use fixed day if specified, else random day in month
            if cat_info.get("day_of_month") == 1:
                txn_date = date(year, month, 1)
            else:
                txn_date = random_date_in_month(year, month)

            amount = generate_amount(cat_info["avg_amount"], cat_info["std_dev"])
            vendor = random.choice(cat_info["vendors"])

            transactions.append({
                "date":        txn_date,
                "description": vendor,
                "amount":      amount,
                "category":    cat_name,
                "type":        "debit"
            })

print(f"✅ Generated {len(transactions)} transactions")


✅ Generated 820 transactions


## Step 4 — Convert to a pandas DataFrame

A **DataFrame** is like an Excel spreadsheet inside Python — rows and columns of data.
We convert our list of dictionaries into a DataFrame, then sort by date.


In [5]:
# pd.DataFrame() takes a list of dicts and turns each dict into one row
df = pd.DataFrame(transactions)

# Convert 'date' column from Python date objects to pandas datetime
# This unlocks powerful time-series features later (groupby month, rolling averages, etc.)
df['date'] = pd.to_datetime(df['date'])

# Sort chronologically
df = df.sort_values('date').reset_index(drop=True)

# Add a month label column — very useful for grouping later
df['month'] = df['date'].dt.to_period('M')   # e.g. "2024-01", "2024-02"

# Peek at the first few rows
print(f"DataFrame shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)


DataFrame shape: 820 rows × 6 columns


,date,description,amount,category,type,month
0,2024-01-01,Salary Credit — Connectivity Solutions,65000,Income,credit,2024-01
1,2024-01-01,Housing Society,18000,Rent,debit,2024-01
2,2024-01-01,Housing Society,18000,Rent,debit,2024-01
3,2024-01-01,BigBasket,710,Groceries,debit,2024-01
4,2024-01-03,Spotify,220,Subscriptions,debit,2024-01
5,2024-01-03,Ola,90,Transport,debit,2024-01
6,2024-01-03,Uber,100,Transport,debit,2024-01
7,2024-01-03,More Supermarket,1120,Groceries,debit,2024-01
8,2024-01-03,Cafe Coffee Day,290,Coffee & Snacks,debit,2024-01
9,2024-01-04,Starbucks,90,Coffee & Snacks,debit,2024-01


## Step 5 — Quick sanity check

Before saving data, always do a quick check:
- Does every category exist?
- Are amounts reasonable?
- Any missing values?


In [6]:
print("=" * 50)
print("DATA QUALITY CHECK")
print("=" * 50)

# Missing values?
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Amount range
print(f"\nAmount range: ₹{df['amount'].min():,.0f}  →  ₹{df['amount'].max():,.0f}")

# Transactions per category
print(f"\nTransactions per category:")
print(df.groupby('category')['amount'].agg(['count', 'sum', 'mean']).round(0).to_string())


DATA QUALITY CHECK

Missing values per column:
date           0
description    0
amount         0
category       0
type           0
month          0
dtype: int64

Amount range: ₹40  →  ₹65,000

Transactions per category:
                  count     sum     mean
category                                
Coffee & Snacks     166   30170    182.0
Entertainment        23   15760    685.0
Food & Dining       146   51010    349.0
Groceries            97   80890    834.0
Health & Fitness     31   17540    566.0
Income               12  780000  65000.0
Rent                 17  306000  18000.0
Shopping             35   41210   1177.0
Subscriptions        48    9430    196.0
Transport           221   48700    220.0
Utilities            24   20510    855.0


## Step 6 — Monthly financial summary

Let's compute total income vs total expenses each month.
This is the kind of insight the Streamlit app will display later.


In [7]:
# Separate income and expenses
income_df  = df[df['type'] == 'credit']
expense_df = df[df['type'] == 'debit']

monthly_income  = income_df.groupby('month')['amount'].sum()
monthly_expense = expense_df.groupby('month')['amount'].sum()

summary = pd.DataFrame({
    'Income (₹)':   monthly_income,
    'Expenses (₹)': monthly_expense,
})
summary['Savings (₹)']  = summary['Income (₹)'] - summary['Expenses (₹)']
summary['Savings Rate'] = (summary['Savings (₹)'] / summary['Income (₹)'] * 100).round(1).astype(str) + '%'

print("Monthly Financial Summary — 2024")
print(summary.to_string())


Monthly Financial Summary — 2024
         Income (₹)  Expenses (₹)  Savings (₹) Savings Rate
month                                                      
2024-01       65000         61310         3690         5.7%
2024-02       65000         44970        20030        30.8%
2024-03       65000         45180        19820        30.5%
2024-04       65000         47360        17640        27.1%
2024-05       65000         43700        21300        32.8%
2024-06       65000         61330         3670         5.6%
2024-07       65000         60950         4050         6.2%
2024-08       65000         41390        23610        36.3%
2024-09       65000         62860         2140         3.3%
2024-10       65000         65080          -80        -0.1%
2024-11       65000         41290        23710        36.5%
2024-12       65000         45800        19200        29.5%


## Step 7 — Save to CSV

We save the full transactions table as `transactions.csv`.
This file will be read by every subsequent notebook in this project.


In [8]:
import os
os.makedirs('data', exist_ok=True)   # create a 'data/' folder if it doesn't exist

df.to_csv('data/transactions.csv', index=False)

print("✅ Saved to data/transactions.csv")
print(f"   File contains {len(df)} rows")
print(f"   Columns: {list(df.columns)}")
print()
print("Next step → Phase 2: Exploratory Data Analysis (EDA)")


✅ Saved to data/transactions.csv
   File contains 820 rows
   Columns: ['date', 'description', 'amount', 'category', 'type', 'month']

Next step → Phase 2: Exploratory Data Analysis (EDA)
